# SQL reference

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/sql/sql-reference.ipynb)

SQL is the shared language of every warehouse and lakehouse engine, and the most predictable part of a data-engineering interview. This notebook covers the query patterns real pipelines use (joins, window functions, CTEs, deduplication, upserts, SCD2) and the plan-reading skills that make performance explainable. Examples run in-process on DuckDB and SQLite.

Sections marked *(engine-specific)* describe behaviour that differs between database engines. The **Dialect differences** table is the reference for syntax across Postgres, MySQL, SQL Server, Snowflake and BigQuery.

In [ ]:
# Setup: installs this notebook's packages when they are missing (e.g. in Google Colab).
import importlib.util
import subprocess
import sys

PACKAGES = {"duckdb": "duckdb>=1.0"}


def _missing(module: str, spec: str) -> bool:
    """True when the module is absent or its installed version does not satisfy spec."""
    try:
        if importlib.util.find_spec(module) is None:
            return True
    except ModuleNotFoundError:
        return True
    try:
        from importlib.metadata import PackageNotFoundError, version

        from packaging.requirements import Requirement
    except ImportError:
        return False
    requirement = Requirement(spec)
    try:
        return not requirement.specifier.contains(version(requirement.name), prereleases=True)
    except PackageNotFoundError:
        return False


missing = [spec for module, spec in PACKAGES.items() if _missing(module, spec)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("ready:", ", ".join(PACKAGES))

## Sample data

One in-memory DuckDB database is shared by the examples below. `q()` runs a query and returns plain Python tuples so cells can `assert` on the result.

In [ ]:
import duckdb

con = duckdb.connect()
con.execute("""
CREATE TABLE departments (dept_id INTEGER PRIMARY KEY, dept_name VARCHAR);
INSERT INTO departments VALUES (1, 'Marketing'), (2, 'IT'), (3, 'Finance');

CREATE TABLE employees (
    emp_id INTEGER PRIMARY KEY, name VARCHAR, email VARCHAR,
    dept_id INTEGER, manager_id INTEGER, salary INTEGER, hire_date DATE
);
INSERT INTO employees VALUES
    (1, 'Ada',    'ada@x.io',    2, NULL, 9000, '2019-04-01'),
    (2, 'Brett',  'brett@x.io',  2, 1,    5000, '2021-01-15'),
    (3, 'Chen',   'chen@x.io',   2, 1,    7000, '2022-07-01'),
    (4, 'Dana',   'dana@x.io',   3, 1,    7000, '2020-02-20'),
    (5, 'Eli',    'eli@x.io',    3, 4,    3000, '2023-03-10'),
    (6, 'Brett',  'brett@x.io',  3, 4,    5000, '2024-05-05');

CREATE TABLE customers (customer_id INTEGER PRIMARY KEY, name VARCHAR, country VARCHAR);
INSERT INTO customers VALUES (1, 'Acme', 'US'), (2, 'Globex', 'DE'), (3, 'Initech', 'US');

CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY, customer_id INTEGER, order_date DATE,
    amount DECIMAL(10, 2), status VARCHAR
);
INSERT INTO orders VALUES
    (10, 1, '2026-01-01', 100, 'completed'),
    (11, 1, '2026-01-02', 250, 'completed'),
    (12, 1, '2026-01-02',  50, 'cancelled'),
    (13, 2, '2026-01-03', 300, 'completed'),
    (14, 4, '2026-01-04',  75, 'completed');   -- customer 4 does not exist
""")


def q(sql, params=None):
    return con.execute(sql, params or []).fetchall()


print({t: q(f"SELECT count(*) FROM {t}")[0][0] for t in ["departments", "employees", "customers", "orders"]})

## Query fundamentals

### Logical execution order: WHERE versus HAVING

SQL is evaluated `FROM`/`JOIN` -> `WHERE` -> `GROUP BY` -> `HAVING` -> `SELECT` -> `ORDER BY` -> `LIMIT`. `WHERE` filters rows before grouping and can use an index; `HAVING` filters groups after aggregation and can reference aggregates. Filtering an aggregate in `WHERE` is an error; moving a row filter into `HAVING` is legal but aggregates far more rows than necessary. A `SELECT` alias is not visible in `WHERE` for the same reason (some engines relax this in `GROUP BY`/`HAVING`).

In [ ]:
rows = q("""
    SELECT dept_id, count(*) AS headcount, avg(salary) AS avg_salary
    FROM employees
    WHERE hire_date >= DATE '2020-01-01'   -- rows, before grouping
    GROUP BY dept_id
    HAVING count(*) >= 2                   -- groups, after aggregation
    ORDER BY dept_id
""")
print(rows)
assert rows == [(2, 2, 6000.0), (3, 3, 5000.0)]

### NULL and three-valued logic

`NULL` means unknown, not zero or blank. Every comparison with `NULL` yields `UNKNOWN`, and `WHERE`/`CASE` keep only `TRUE`, so `= NULL` and `<> NULL` match nothing: use `IS [NOT] NULL`, `COALESCE`, `NULLIF`, or `IS [NOT] DISTINCT FROM`. `TRUE OR NULL` is `TRUE` and `FALSE AND NULL` is `FALSE`; everything else touching `NULL` is `UNKNOWN`. Two traps follow:

- `NOT IN (subquery)` returns no rows if the subquery yields a single `NULL`. Prefer `NOT EXISTS`.
- `COUNT(*)` counts rows, `COUNT(col)` skips `NULL`s, so they differ by exactly the number of missing values.

In [ ]:
assert q("SELECT CASE WHEN NULL = NULL THEN 'T' ELSE 'F' END")[0][0] == "F"
assert q("SELECT CASE WHEN NULL IS NULL THEN 'T' ELSE 'F' END")[0][0] == "T"

# COUNT(*) vs COUNT(column): manager_id is NULL for Ada
total, non_null = q("SELECT count(*), count(manager_id) FROM employees")[0]
assert (total, non_null) == (6, 5)

# NOT IN with a NULL in the subquery returns nothing; NOT EXISTS behaves
not_in = q("SELECT emp_id FROM employees WHERE emp_id NOT IN (SELECT manager_id FROM employees)")
not_exists = q("""
    SELECT emp_id FROM employees e
    WHERE NOT EXISTS (SELECT 1 FROM employees m WHERE m.manager_id = e.emp_id)
    ORDER BY emp_id
""")
print("NOT IN:", not_in, "| NOT EXISTS:", not_exists)
assert not_in == [] and [r[0] for r in not_exists] == [2, 3, 5, 6]

### CASE: SQL's if/else

`CASE` evaluates branches in order and returns the first match, so put the most specific condition first; without `ELSE` the result is `NULL`. It works in `SELECT`, `WHERE`, `ORDER BY` and inside aggregates, which is how conditional aggregation (pivoting) is written. Oracle's `DECODE` is an equality-only shorthand.

In [ ]:
rows = q("""
    SELECT
        sum(CASE WHEN status = 'completed' THEN amount ELSE 0 END) AS completed,
        sum(CASE WHEN status = 'cancelled' THEN amount ELSE 0 END) AS cancelled,
        count(*) FILTER (WHERE amount >= 100)                     AS big_orders
    FROM orders
""")
print(rows)
assert rows == [(725, 50, 3)]

### Joins

- **Inner**: only matching pairs; the cheapest because non-matches are discarded early.
- **Left / right outer**: preserve every row of one side, padding the other with `NULL`. A right join is a left join with the tables swapped; most teams standardise on `LEFT JOIN`.
- **Full outer**: preserve unmatched rows from both sides (MySQL lacks it; emulate with `LEFT JOIN ... UNION ALL` the unmatched right rows).
- **Cross**: every pair; only deliberate for dimension expansion (date x product).
- **Semi-join** (`EXISTS`): rows with at least one match, never duplicated. **Anti-join** (`NOT EXISTS` or `LEFT JOIN ... WHERE right.key IS NULL`): rows with no match. `EXCEPT` (Oracle `MINUS`) also works but deduplicates.
- **Self-join**: a table joined to itself under two aliases (employee to manager).

In [ ]:
inner = q("SELECT c.name, o.order_id FROM customers c JOIN orders o USING (customer_id) ORDER BY 2")
left = q("SELECT c.name, count(o.order_id) FROM customers c LEFT JOIN orders o USING (customer_id) GROUP BY 1 ORDER BY 1")
full = q("SELECT c.customer_id, o.order_id FROM customers c FULL JOIN orders o USING (customer_id) "
         "WHERE c.customer_id IS NULL OR o.order_id IS NULL ORDER BY 2 NULLS FIRST")
semi = q("SELECT name FROM customers c WHERE EXISTS (SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id) ORDER BY 1")
anti = q("SELECT name FROM customers c WHERE NOT EXISTS (SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id)")
self_join = q("SELECT e.name, m.name FROM employees e LEFT JOIN employees m ON m.emp_id = e.manager_id WHERE e.emp_id IN (1, 5) ORDER BY e.emp_id")

print("inner    :", inner)
print("left     :", left)
print("full, unmatched only:", full)
print("semi/anti:", semi, anti)
print("self     :", self_join)
assert len(inner) == 4                                   # order 14 has no customer
assert left == [("Acme", 3), ("Globex", 1), ("Initech", 0)]  # count(o.order_id) ignores NULL padding
assert full == [(3, None), (None, 14)]
assert [r[0] for r in semi] == ["Acme", "Globex"] and anti == [("Initech",)]
assert self_join == [("Ada", None), ("Eli", "Dana")]

The classic left-join bug: a filter on the right table in `WHERE` discards the `NULL`-padded rows and silently turns the left join into an inner join. Put right-side conditions in `ON`.

In [ ]:
in_where = q("""
    SELECT c.name, o.order_id FROM customers c
    LEFT JOIN orders o ON o.customer_id = c.customer_id
    WHERE o.status = 'completed'
""")
in_on = q("""
    SELECT c.name, o.order_id FROM customers c
    LEFT JOIN orders o ON o.customer_id = c.customer_id AND o.status = 'completed'
""")
print(len(in_where), "rows with the filter in WHERE;", len(in_on), "rows with it in ON")
assert "Initech" not in {r[0] for r in in_where}
assert "Initech" in {r[0] for r in in_on}

### Set operations

Set operators combine result sets with the same number of compatible columns. `UNION` deduplicates (a sort or hash of the whole result); `UNION ALL` just concatenates; `INTERSECT` keeps rows in both; `EXCEPT`/`MINUS` subtracts. In ETL the inputs are usually disjoint by construction, so `UNION ALL` is the right default and `UNION` is wasted work that can also hide a duplicated key you wanted to catch.

### Subqueries and CTEs

A scalar subquery returns one value; a correlated subquery references the outer row and is conceptually evaluated per row (optimisers often decorrelate it into a join). A window function or a join to a pre-aggregated CTE usually does the same work in one pass.

A CTE (`WITH name AS (...)`) is a named subquery for readability and reuse. Whether it is inlined or materialised is an engine decision: Postgres 12+ inlines by default (`AS MATERIALIZED` forces one evaluation); SQL Server and BigQuery tend to inline too, so a CTE is not a performance boundary by itself.

A **recursive CTE** has an anchor query and a recursive step that runs until it produces no new rows. It replaces hand-written self-joins for hierarchies of unknown depth. Guard against cycles with a depth limit, and index the recursive join key. (`RECURSIVE` is required in Postgres, DuckDB, Snowflake and BigQuery; SQL Server omits it.)

In [ ]:
above_dept_avg = q("""
    SELECT name FROM (
        SELECT name, salary, avg(salary) OVER (PARTITION BY dept_id) AS dept_avg
        FROM employees
    ) WHERE salary > dept_avg ORDER BY name
""")

org = q("""
    WITH RECURSIVE org AS (
        SELECT emp_id, name, manager_id, 0 AS depth, name AS path
        FROM employees WHERE manager_id IS NULL
        UNION ALL
        SELECT e.emp_id, e.name, e.manager_id, o.depth + 1, o.path || ' > ' || e.name
        FROM employees e JOIN org o ON e.manager_id = o.emp_id
        WHERE o.depth < 10                        -- cycle guard
    )
    SELECT depth, path FROM org ORDER BY depth, path
""")
print(above_dept_avg)
for depth, path in org:
    print(depth, path)
assert above_dept_avg == [("Ada",), ("Dana",)]
assert (2, "Ada > Dana > Eli") in org and max(d for d, _ in org) == 2

## Classic interview queries

### Second and nth highest salary

Clarify the definition first. `DENSE_RANK` returns the nth distinct salary (ties do not consume a rank); `ROW_NUMBER` returns the nth row even when it ties the one above; `RANK` skips numbers after a tie. The older forms still work: `SELECT max(salary) FROM t WHERE salary < (SELECT max(salary) FROM t)` for the second, or a correlated `count(DISTINCT ...)` for the nth, which rescans the table per row.

In [ ]:
def nth_highest(n):
    return q("""
        SELECT DISTINCT salary FROM (
            SELECT salary, dense_rank() OVER (ORDER BY salary DESC) AS rnk FROM employees
        ) WHERE rnk = ?
    """, [n])


second_subquery = q("SELECT max(salary) FROM employees WHERE salary < (SELECT max(salary) FROM employees)")
print("2nd:", nth_highest(2), "3rd:", nth_highest(3), "subquery form:", second_subquery)
assert nth_highest(2) == [(7000,)] and nth_highest(3) == [(5000,)]
assert second_subquery == [(7000,)]

### Max salary per department, with the department name

Two traps: departments with no employees (ask whether they should appear; if so, drive from `departments` with a `LEFT JOIN`), and returning the whole top-earner row, which `max()` cannot do without a second join. A window function returns the full row and controls tie-breaking (`ROW_NUMBER` picks one, `RANK` keeps ties). `QUALIFY` (DuckDB, Snowflake, BigQuery) filters a window result without a subquery.

In [ ]:
max_per_dept = q("""
    SELECT d.dept_name, max(e.salary)
    FROM departments d LEFT JOIN employees e ON e.dept_id = d.dept_id
    GROUP BY d.dept_name ORDER BY d.dept_name
""")
top_earner = q("""
    SELECT dept_id, name, salary FROM employees
    QUALIFY rank() OVER (PARTITION BY dept_id ORDER BY salary DESC) = 1
    ORDER BY dept_id
""")
print(max_per_dept)
print(top_earner)
assert max_per_dept == [("Finance", 7000), ("IT", 9000), ("Marketing", None)]
assert top_earner == [(2, "Ada", 9000), (3, "Dana", 7000)]

### Find and remove duplicates

Group by the columns that define a duplicate and keep groups with `count(*) > 1`. To delete all but one row, rank within the duplicate key and delete `rn > 1`; the ordering decides the survivor (latest `updated_at`, most trusted source), and a final tiebreaker makes it deterministic. MySQL refuses to delete from a table its own subquery reads unless the ids are wrapped in a derived table or staged first.

In [ ]:
dupes = q("SELECT name, email, count(*) FROM employees GROUP BY name, email HAVING count(*) > 1")
print("duplicates:", dupes)
assert dupes == [("Brett", "brett@x.io", 2)]

con.execute("CREATE TEMP TABLE emp_dedup AS SELECT * FROM employees")
con.execute("""
    DELETE FROM emp_dedup WHERE emp_id IN (
        SELECT emp_id FROM (
            SELECT emp_id, row_number() OVER (PARTITION BY email ORDER BY hire_date DESC, emp_id DESC) AS rn
            FROM emp_dedup
        ) WHERE rn > 1
    )
""")
survivor = q("SELECT emp_id FROM emp_dedup WHERE email = 'brett@x.io'")
assert survivor == [(6,)] and q("SELECT count(*) FROM emp_dedup")[0][0] == 5
print("kept the newest Brett:", survivor)

### Pagination with a total count

`count(*) OVER ()` returns the total alongside each page in one query (MySQL's old `SQL_CALC_FOUND_ROWS` is deprecated). On large results the window forces a full scan, so a separate or approximate count can be cheaper. Large `OFFSET`s get slower with every page because skipped rows are still read; **keyset pagination** (`WHERE (sort_key, id) < (:last_key, :last_id)`) seeks directly. Dialects: `LIMIT n OFFSET m` (Postgres, MySQL, DuckDB), `TOP n` / `OFFSET ... FETCH` (SQL Server), `ROWNUM` / `FETCH FIRST` (Oracle).

In [ ]:
page = q("SELECT count(*) OVER () AS total, emp_id FROM employees ORDER BY salary DESC, emp_id LIMIT 2 OFFSET 2")
keyset = q("SELECT emp_id FROM employees WHERE (salary, emp_id) < (7000, 4) ORDER BY salary DESC, emp_id DESC LIMIT 2")
print(page, keyset)
assert page == [(6, 4), (6, 2)] and keyset == [(3,), (6,)]

### Dates, ranges and pattern matching

- Quarter: `extract(quarter FROM d)` (Postgres, Snowflake, BigQuery, DuckDB), `quarter(d)` (MySQL), `to_char(d, 'Q')` (Oracle). `date_trunc('quarter', d)` returns the first day, which is what a date-dimension join wants.
- `BETWEEN` is inclusive on both ends. On a timestamp column `BETWEEN '2000-01-01' AND '2000-12-31'` misses everything after midnight on the last day; prefer a half-open range `>= start AND < next_start`.
- Use single quotes for literals; double quotes are identifiers in ANSI SQL.
- Case-insensitive search: `lower(name) LIKE '%rich%'` or `ILIKE` (Postgres, Snowflake, DuckDB). A leading wildcard cannot use a B-tree index; use trigram or full-text indexes.

In [ ]:
import datetime

quarter, quarter_start = q("SELECT extract(quarter FROM DATE '2026-03-31'), "
                           "date_trunc('quarter', DATE '2026-05-15')::DATE")[0]
assert (quarter, quarter_start) == (1, datetime.date(2026, 4, 1))
con.execute("CREATE TEMP TABLE ev AS SELECT TIMESTAMP '2000-12-31 15:00:00' AS ts")
between = q("SELECT count(*) FROM ev WHERE ts BETWEEN '2000-01-01' AND '2000-12-31'")[0][0]
half_open = q("SELECT count(*) FROM ev WHERE ts >= '2000-01-01' AND ts < '2001-01-01'")[0][0]
print("BETWEEN:", between, "half-open:", half_open)
assert (between, half_open) == (0, 1)

## Window functions

A window function computes over related rows without collapsing them the way `GROUP BY` does. The spec has three parts: `PARTITION BY` (the groups), `ORDER BY` (sequence within a group) and the **frame** (which rows each output row sees). Data-engineering uses: deduplication, latest record per key, running totals, moving averages, ranking, change detection (`LAG`/`LEAD`) and sessionization.

### Ranking functions

`ROW_NUMBER` never repeats (ties broken arbitrarily unless the `ORDER BY` is unique); `RANK` repeats on ties and skips (1, 2, 2, 4); `DENSE_RANK` repeats without skipping (1, 2, 2, 3). Use `ROW_NUMBER` for dedup and top-1; `RANK`/`DENSE_RANK` when ties matter. `NTILE(n)` buckets rows into n groups.

In [ ]:
ranks = q("""
    SELECT name, salary,
           row_number() OVER (ORDER BY salary DESC, emp_id) AS rn,
           rank()       OVER (ORDER BY salary DESC)         AS rnk,
           dense_rank() OVER (ORDER BY salary DESC)         AS drnk,
           ntile(3)     OVER (ORDER BY salary DESC, emp_id) AS tercile
    FROM employees
    ORDER BY rn
""")
print("name, salary, row_number, rank, dense_rank, ntile(3)")
for row in ranks:
    print(row)
assert [r[2] for r in ranks] == [1, 2, 3, 4, 5, 6]
assert [r[3] for r in ranks] == [1, 2, 2, 4, 4, 6]
assert [r[4] for r in ranks] == [1, 2, 2, 3, 3, 4]

### Frames: ROWS versus RANGE

With an `ORDER BY` and no explicit frame, the default is `RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW`, which includes **peer** rows that share the ordering value. With duplicate dates a "running total" therefore jumps by both rows at once. Write `ROWS` when you mean one physical row at a time, and add a unique tiebreaker to the `ORDER BY`. Common frames:

- `ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW`: running total.
- `ROWS BETWEEN 6 PRECEDING AND CURRENT ROW`: seven-row moving window.
- `ROWS BETWEEN 1 PRECEDING AND 1 FOLLOWING`: centred smoothing.
- `ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING`: the whole partition.

In [ ]:
frames = q("""
    SELECT order_id, order_date, amount,
           sum(amount) OVER (ORDER BY order_date) AS range_default,
           sum(amount) OVER (ORDER BY order_date, order_id
                             ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS rows_total,
           avg(amount) OVER (ORDER BY order_date, order_id
                             ROWS BETWEEN 1 PRECEDING AND CURRENT ROW) AS moving_avg_2
    FROM orders ORDER BY order_date, order_id
""")
for row in frames:
    print(row)
by_id = {r[0]: r for r in frames}
assert by_id[11][3] == by_id[12][3] == 400       # RANGE: both 2026-01-02 rows see each other
assert (by_id[11][4], by_id[12][4]) == (350, 400)  # ROWS: one row at a time

### LAG/LEAD, change detection and gaps-and-islands

`LAG(col)` reads the previous row in the partition and `LEAD(col)` the next one, which turns "did the value change?" into a row-level comparison. **Sessionization** is the gaps-and-islands pattern: flag a row as a new session when the gap to the previous event exceeds a threshold, then take a running sum of the flags to number the sessions.

In [ ]:
con.execute("""
CREATE TEMP TABLE clicks AS SELECT * FROM (VALUES
    ('u1', TIMESTAMP '2026-01-01 10:00'), ('u1', TIMESTAMP '2026-01-01 10:10'),
    ('u1', TIMESTAMP '2026-01-01 11:30'), ('u1', TIMESTAMP '2026-01-01 11:35'),
    ('u2', TIMESTAMP '2026-01-01 09:00')
) AS t(user_id, ts)
""")
sessions = q("""
    WITH flagged AS (
        SELECT user_id, ts,
               CASE WHEN ts - lag(ts) OVER (PARTITION BY user_id ORDER BY ts) <= INTERVAL 30 MINUTE
                    THEN 0 ELSE 1 END AS new_session
        FROM clicks
    )
    SELECT user_id, ts,
           sum(new_session) OVER (PARTITION BY user_id ORDER BY ts
                                  ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS session_no
    FROM flagged ORDER BY user_id, ts
""")
for row in sessions:
    print(row)
assert [r[2] for r in sessions] == [1, 1, 2, 2, 1]

### Top-N per group

`ROW_NUMBER() ... WHERE rn <= N` in a subquery, or `QUALIFY` where supported (Snowflake, BigQuery, DuckDB, Teradata). Engines can often turn this into a per-partition top-N heap instead of a full sort. Use `RANK` if ties at the cut-off should all be kept.

## Data-engineering SQL patterns

### Deduplication

Define the business key, order duplicates by trust or recency, keep one winner. Avoid `SELECT DISTINCT` unless every column must match, because real duplicates differ in ingestion time, source file or metadata. In production, write rejected duplicates to an audit table or track the duplicate rate as a data-quality metric. For late-arriving duplicates (mobile retries up to 72 hours late), reprocess only that bounded window.

In [ ]:
con.execute("""
CREATE TEMP TABLE raw_orders AS SELECT * FROM (VALUES
    (1, 'new',     TIMESTAMP '2026-01-01 10:00', 1),
    (1, 'shipped', TIMESTAMP '2026-01-02 10:00', 2),
    (1, 'shipped', TIMESTAMP '2026-01-02 10:00', 3),   -- exact replay of the same update
    (2, 'new',     TIMESTAMP '2026-01-01 12:00', 4)
) AS t(order_id, status, updated_at, ingest_seq)
""")
latest = q("""
    SELECT order_id, status FROM raw_orders
    QUALIFY row_number() OVER (PARTITION BY order_id ORDER BY updated_at DESC, ingest_seq DESC) = 1
    ORDER BY order_id
""")
print(latest)
assert latest == [(1, "shipped"), (2, "new")]

### Upserts: MERGE and INSERT ... ON CONFLICT (engine-specific)

An upsert inserts new keys and updates existing ones atomically, avoiding the read-then-write race of `SELECT` followed by `INSERT`/`UPDATE`. Spellings: `MERGE` (SQL Server, Oracle, Snowflake, BigQuery, Postgres 15+, DuckDB 1.4+), `INSERT ... ON CONFLICT DO UPDATE` (Postgres, SQLite, DuckDB), `INSERT ... ON DUPLICATE KEY UPDATE` (MySQL).

```sql
MERGE INTO analytics.customers AS t
USING staging.customers AS s ON t.customer_id = s.customer_id
WHEN MATCHED THEN UPDATE SET email = s.email, updated_at = s.updated_at
WHEN NOT MATCHED THEN INSERT (customer_id, email, updated_at)
                      VALUES (s.customer_id, s.email, s.updated_at);
```

Deduplicate the source first: a `MERGE` source with duplicate keys errors in BigQuery and Snowflake and may update twice elsewhere. For append-only facts a plain insert is cheaper than an update-heavy merge.

In [ ]:
con.execute("CREATE TEMP TABLE dim_cust (customer_id INTEGER PRIMARY KEY, email VARCHAR, updated_at DATE)")
con.execute("INSERT INTO dim_cust VALUES (1, 'old@acme.io', '2026-01-01')")
con.execute("""
    INSERT INTO dim_cust VALUES (1, 'new@acme.io', '2026-02-01'), (2, 'hi@globex.io', '2026-02-01')
    ON CONFLICT (customer_id) DO UPDATE SET email = EXCLUDED.email, updated_at = EXCLUDED.updated_at
""")
result = q("SELECT customer_id, email FROM dim_cust ORDER BY 1")
print(result)
assert result == [(1, "new@acme.io"), (2, "hi@globex.io")]

### Incremental models

An incremental model processes only new or changed rows. It needs a reliable watermark (`updated_at`, ingestion time, source offset or partition date) and a write strategy: `MERGE` on the key, partition overwrite, or append. The main risk is late-arriving data: reprocess a lookback window (for example the last 2-7 days) and deduplicate by key, so reruns are idempotent.

In [ ]:
con.execute("CREATE TEMP TABLE stg AS SELECT * FROM (VALUES (1, 10, DATE '2026-01-01'), (2, 20, DATE '2026-01-02')) t(id, amount, updated_at)")
con.execute("CREATE TEMP TABLE tgt (id INTEGER PRIMARY KEY, amount INTEGER, updated_at DATE)")

INCREMENTAL = """
    INSERT INTO tgt
    SELECT * FROM stg
    WHERE updated_at > (SELECT coalesce(max(updated_at), DATE '1900-01-01') - INTERVAL 1 DAY FROM tgt)  -- 1-day lookback
    ON CONFLICT (id) DO UPDATE SET amount = EXCLUDED.amount, updated_at = EXCLUDED.updated_at
"""
con.execute(INCREMENTAL)
con.execute("INSERT INTO stg VALUES (2, 25, DATE '2026-01-02'), (3, 30, DATE '2026-01-03')")  # late correction + new row
con.execute("DELETE FROM stg WHERE id = 2 AND amount = 20")
con.execute(INCREMENTAL)
con.execute(INCREMENTAL)  # rerun is a no-op: idempotent
final = q("SELECT * FROM tgt ORDER BY id")
print(final)
assert [(r[0], r[1]) for r in final] == [(1, 10), (2, 25), (3, 30)]

### SCD Type 2 and point-in-time joins

Slowly changing dimensions answer "what was this customer's segment when the order happened?". **Type 1** overwrites (no history); **Type 2** keeps one row per version with `valid_from`, `valid_to` and `is_current`; **Type 3** keeps a "previous value" column. A Type 2 load closes the current version of changed keys and inserts the new versions. Facts must then join on the key **and** the validity window; joining on the key alone returns every version and multiplies fact rows. A unique index on `customer_id WHERE is_current` (Postgres partial index) guards against two current rows.

In [ ]:
con.execute("""
CREATE TEMP TABLE dim_customer_scd2 (
    customer_id INTEGER, segment VARCHAR, valid_from DATE, valid_to DATE, is_current BOOLEAN
);
INSERT INTO dim_customer_scd2 VALUES (1, 'SMB', '2025-01-01', NULL, TRUE), (2, 'SMB', '2025-01-01', NULL, TRUE);
CREATE TEMP TABLE stg_changes AS SELECT * FROM (VALUES (1, 'Enterprise'), (2, 'SMB')) t(customer_id, segment);
""")
load_date = "2026-01-02"
# 1. close current versions whose attributes changed
con.execute("""
    UPDATE dim_customer_scd2 d SET valid_to = ?::DATE, is_current = FALSE
    FROM stg_changes s
    WHERE d.customer_id = s.customer_id AND d.is_current AND d.segment <> s.segment
""", [load_date])
# 2. insert a new current version for every key that no longer has one
con.execute("""
    INSERT INTO dim_customer_scd2
    SELECT s.customer_id, s.segment, ?::DATE, NULL, TRUE FROM stg_changes s
    WHERE NOT EXISTS (SELECT 1 FROM dim_customer_scd2 d WHERE d.customer_id = s.customer_id AND d.is_current)
""", [load_date])

point_in_time = q("""
    SELECT o.order_id, o.order_date, c.segment
    FROM (VALUES (100, 1, DATE '2026-01-01'), (101, 1, DATE '2026-01-05')) o(order_id, customer_id, order_date)
    JOIN dim_customer_scd2 c
      ON c.customer_id = o.customer_id
     AND o.order_date >= c.valid_from
     AND o.order_date <  coalesce(c.valid_to, DATE '9999-12-31')
    ORDER BY o.order_id
""")
print(q("SELECT * FROM dim_customer_scd2 ORDER BY customer_id, valid_from"))
print(point_in_time)
assert q("SELECT count(*) FROM dim_customer_scd2")[0][0] == 3          # customer 2 unchanged: no new version
assert [r[2] for r in point_in_time] == ["SMB", "Enterprise"]

### GROUPING SETS, ROLLUP and CUBE

These compute several grouping levels in one pass. `ROLLUP (a, b)` gives `(a, b)`, `(a)` and the grand total; `CUBE` gives every combination; `GROUPING SETS` lists exactly the levels you want. `GROUPING(col)` tells a real `NULL` apart from a subtotal row.

In [ ]:
rollup = q("""
    SELECT c.country, o.status, sum(o.amount) AS revenue, grouping(c.country, o.status) AS lvl
    FROM orders o JOIN customers c USING (customer_id)
    GROUP BY ROLLUP (c.country, o.status)
    ORDER BY lvl, c.country, o.status
""")
for row in rollup:
    print(row)
assert rollup[-1] == (None, None, 700, 3)   # grand total

### Querying JSON (engine-specific)

Engines query semi-structured columns in place: parse, navigate a path, cast to a typed scalar.

```sql
SELECT payload ->> 'event_type', (payload -> 'amount')::numeric FROM events;  -- Postgres jsonb
SELECT payload:event_type::string FROM events;                                -- Snowflake VARIANT
SELECT JSON_VALUE(payload, '$.event_type') FROM events;                       -- BigQuery
```

Path extraction is a function on the column, so it is not sargable by default and reads the whole document. Extract hot paths into typed columns at load time, or build an expression index on the path.

### Dialect differences (engine-specific)

| Concern | Postgres | MySQL | SQL Server | Snowflake | BigQuery |
| --- | --- | --- | --- | --- | --- |
| Limit rows | `LIMIT n` | `LIMIT n` | `TOP n` / `OFFSET FETCH` | `LIMIT n` | `LIMIT n` |
| Null-safe equal | `IS NOT DISTINCT FROM` | `<=>` | `IS NOT DISTINCT FROM` (2022+) | `EQUAL_NULL` | `IS NOT DISTINCT FROM` |
| Upsert | `ON CONFLICT` / `MERGE` | `ON DUPLICATE KEY` | `MERGE` | `MERGE` | `MERGE` |
| Identifier quotes | `"id"` | `` `id` `` | `[id]` | `"id"` | `` `id` `` |
| Unquoted identifier case | folded to lower | as written | as written | folded to upper | case-sensitive |
| Non-aggregated column in GROUP BY | error | allowed (legacy mode) | error | error | error |

Write portable SQL with unquoted lower-case names, explicit casts, and every non-aggregated column in the `GROUP BY`.

## Schema design and constraints

### Keys

- **Primary key**: `NOT NULL` + `UNIQUE`, one per table; in InnoDB and SQL Server usually the clustered index.
- **Unique key**: enforces uniqueness on an alternate key; `NULL`s allowed (one or many, depending on the engine).
- **Foreign key**: a value that must exist in the referenced table, with `ON DELETE` / `ON UPDATE` actions (`CASCADE`, `RESTRICT`, `SET NULL`).
- **Candidate key**: any minimal column set that uniquely identifies a row. The chosen one is the primary key; the rest are **alternate keys**. A **composite key** spans several columns (typical for junction tables).

Prefer a stable surrogate primary key plus a unique constraint on the natural business key: emails change, and a changing primary key cascades to every child row.

### Constraints and relationships

`NOT NULL`, `DEFAULT`, `CHECK`, `UNIQUE`, `PRIMARY KEY` and `FOREIGN KEY` catch bad data at write time and give the optimiser facts it can trust. Relationships map to tables as follows: one-to-many puts the foreign key on the many side; one-to-one makes the child's primary key also its foreign key; many-to-many uses a junction table with a composite primary key (add an index on the second column if you query from both directions). `ON DELETE CASCADE` is convenient but can delete far more than intended; many teams prefer `RESTRICT` plus an explicit cleanup job. Cloud warehouses (Snowflake, BigQuery, Redshift) accept primary and foreign keys but mostly do not enforce them, so tests in the pipeline (dbt `unique`/`not_null`/`relationships`) do the enforcing.

In [ ]:
import sqlite3

db = sqlite3.connect(":memory:")
db.execute("PRAGMA foreign_keys = ON")      # SQLite enforces FKs only when asked
db.executescript("""
CREATE TABLE products (product_id INTEGER PRIMARY KEY, sku TEXT NOT NULL UNIQUE);
CREATE TABLE orders (order_id INTEGER PRIMARY KEY,
                     amount REAL NOT NULL CHECK (amount >= 0),
                     status TEXT NOT NULL DEFAULT 'new' CHECK (status IN ('new', 'paid', 'cancelled')));
CREATE TABLE order_items (
    order_id   INTEGER NOT NULL REFERENCES orders(order_id) ON DELETE CASCADE,
    product_id INTEGER NOT NULL REFERENCES products(product_id),
    quantity   INTEGER NOT NULL DEFAULT 1,
    PRIMARY KEY (order_id, product_id)                 -- junction table: many-to-many
);
INSERT INTO products VALUES (1, 'A-1'), (2, 'B-2');
INSERT INTO orders (order_id, amount) VALUES (42, 10.0);
INSERT INTO order_items VALUES (42, 1, 2), (42, 2, 1);
""")

violations = {}
for label, sql in {
    "check": "INSERT INTO orders (order_id, amount) VALUES (43, -5)",
    "unique": "INSERT INTO products VALUES (3, 'A-1')",
    "foreign key": "INSERT INTO order_items VALUES (42, 99, 1)",
    "composite pk": "INSERT INTO order_items VALUES (42, 1, 5)",
}.items():
    try:
        db.execute(sql)
    except sqlite3.IntegrityError as exc:
        violations[label] = str(exc)
print(violations)
assert len(violations) == 4

db.execute("DELETE FROM orders WHERE order_id = 42")
assert db.execute("SELECT count(*) FROM order_items").fetchone()[0] == 0
print("ON DELETE CASCADE removed the child rows")

### Normalization and denormalization

Each normal form removes a specific update anomaly:

- **1NF**: atomic values, no repeating groups.
- **2NF**: no non-key column depends on only part of a composite key (`course_name` in an `enrollments(student_id, course_id)` table belongs in `courses`).
- **3NF**: no transitive dependency; a non-key column must not depend on another non-key column.
- **BCNF**: every determinant is a candidate key (stricter 3NF for overlapping candidate keys). **4NF** removes multi-valued dependencies.

Normalise to 3NF for OLTP, where writes must be correct. Denormalise deliberately for reads (star schemas, wide marts, materialized views) because analytical joins over many normalised tables are expensive.

### Views, materialized views and stored procedures

A **view** is a stored query: it exposes a stable interface, hides sensitive columns (`GRANT SELECT` on the view, not the table) and centralises logic. Deeply nested views can hide work from the optimiser and block predicate pushdown. A **materialized view** stores the result and must be refreshed. A **stored procedure** runs multi-statement logic (parameters, branches, transactions) next to the data; it is harder to version and test than SQL models in dbt or Spark, so modern teams keep it for operational tasks.

### Command families and DELETE versus TRUNCATE

DDL defines structure (`CREATE`, `ALTER`, `DROP`, `TRUNCATE`); DML changes data (`INSERT`, `UPDATE`, `DELETE`, `MERGE`); DQL reads (`SELECT`); DCL grants (`GRANT`, `REVOKE`); TCL controls transactions (`COMMIT`, `ROLLBACK`, `SAVEPOINT`). Oracle and MySQL auto-commit DDL; Postgres and SQL Server can roll DDL back inside a transaction.

`DELETE` removes filtered rows one by one, fires triggers, is fully logged and can be rolled back. `TRUNCATE` deallocates all pages at once, is minimally logged and much faster, resets identity counters in MySQL, and fails in SQL Server when a foreign key references the table. Postgres lets you roll back a `TRUNCATE`; Oracle and MySQL do not.

## Transactions and concurrency

### ACID

A transaction is a unit of work that commits entirely or rolls back entirely.

- **Atomicity**: all statements take effect or none do.
- **Consistency**: constraints hold before and after (a transfer cannot create money).
- **Isolation**: concurrent transactions do not see each other's in-flight changes, to the degree the isolation level promises.
- **Durability**: a committed change survives a crash, because the write-ahead log was flushed first.

Isolation is the property teams relax for throughput; durability is never relaxed for money movement.

In [ ]:
bank = sqlite3.connect(":memory:", isolation_level=None)   # explicit BEGIN/COMMIT
bank.executescript("""
CREATE TABLE accounts (id INTEGER PRIMARY KEY, balance INTEGER NOT NULL CHECK (balance >= 0));
INSERT INTO accounts VALUES (1, 100), (2, 0);
""")


def transfer(amount):
    try:
        bank.execute("BEGIN")
        bank.execute("UPDATE accounts SET balance = balance + ? WHERE id = 2", (amount,))
        bank.execute("UPDATE accounts SET balance = balance - ? WHERE id = 1", (amount,))
        bank.execute("COMMIT")
    except sqlite3.IntegrityError:
        bank.execute("ROLLBACK")


transfer(60)    # succeeds
transfer(60)    # would overdraw account 1: the CHECK fails, both updates roll back
balances = bank.execute("SELECT balance FROM accounts ORDER BY id").fetchall()
print(balances)
assert balances == [(40,), (60,)]

### Isolation levels (engine-specific)

| Level | Dirty read | Non-repeatable read | Phantom read |
|---|---|---|---|
| Read uncommitted | possible | possible | possible |
| Read committed | no | possible | possible |
| Repeatable read | no | no | possible (prevented by InnoDB next-key locks and by snapshot engines) |
| Serializable | no | no | no |

Defaults: Postgres, SQL Server and Oracle use read committed; MySQL InnoDB uses repeatable read. Snowflake and BigQuery give snapshot isolation without the same knobs. Higher isolation prevents more anomalies at the cost of more locking or serialization-failure retries. MVCC engines (Postgres, Oracle, InnoDB) let readers see a snapshot without blocking writers.

### Locks, blocking and deadlocks

**Blocking** is one session waiting for a lock another holds; it resolves when the holder commits. Diagnose with the engine's lock views (`pg_locks` + `pg_stat_activity`, `sys.dm_exec_requests`, `SHOW ENGINE INNODB STATUS`), then shorten transactions, add indexes so updates lock fewer rows, and avoid user think-time inside a transaction. A **deadlock** is a cycle of sessions each waiting on the other; the engine detects it and kills one victim, so the application must retry. Prevent it by acquiring locks in a consistent order. A **livelock** is sessions that keep retrying and yielding without making progress.

### Write-ahead log, checkpoints and replication

The **write-ahead log** records every change before the data pages are written, so a crash is recovered by replaying the log (durability) and pages can be flushed lazily. A **checkpoint** flushes dirty pages and marks a log position, bounding how much log recovery must replay. The same log feeds **replication** (streaming/physical, logical, or SQL Server's snapshot, transactional and merge replication) and **change data capture** tools such as Debezium. **Sharding** splits a table horizontally across servers by a shard key; it scales writes but makes cross-shard joins and transactions expensive, so choose a key that keeps related rows together.

### SQL injection

Concatenating user input into SQL lets an attacker change the statement. Bound parameters are the only reliable defence; escaping and blacklists are not. Table and column names cannot be parameterised, so validate them against an allowlist, and grant the application role least privilege.

In [ ]:
users = sqlite3.connect(":memory:")
users.executescript("CREATE TABLE users (email TEXT, is_admin INTEGER); "
                    "INSERT INTO users VALUES ('alice@x.io', 0), ('root@x.io', 1);")
attack = "nobody' OR '1'='1"

unsafe = users.execute("SELECT email FROM users WHERE email = '" + attack + "'").fetchall()
safe = users.execute("SELECT email FROM users WHERE email = ?", (attack,)).fetchall()
print("concatenated:", unsafe, "| bound parameter:", safe)
assert len(unsafe) == 2 and safe == []

## Query plans

SQL says *what* you want; the plan says *how* the engine gets it. A plan is a tree of operators: leaves read tables or indexes, and rows flow up through filters, joins, aggregates and sorts to the root.

- **Scan** (Seq Scan, table access full): read every row.
- **Seek / index scan / SEARCH**: descend a B-tree and read only a key range.
- **Filter**: drop rows after reading them.
- **Join**: nested loop, hash or merge, with one side as the outer/build input.
- **Aggregate**: hash table or sorted run.
- **Sort**: the usual source of memory pressure and spills.
- **Exchange / shuffle**: move rows between workers in distributed engines.

`EXPLAIN` shows the plan the optimiser would choose without running it. `EXPLAIN ANALYZE` (Postgres, MySQL 8, SQL Server's actual plan) runs the statement and reports real row counts and timings next to the estimates. It really executes, so wrap DML in a transaction and roll back. Snowflake's Query Profile and BigQuery's execution details are the warehouse equivalents.

### What to look for

1. Find the operator with the most time, not the scariest name.
2. Compare estimated with actual rows. The first operator where they diverge by orders of magnitude explains bad choices above it (stale statistics, skew, correlated columns, functions on columns).
3. Full scans of big tables where a selective predicate should have used an index or partition pruning.
4. Nested loops over a large inner input; hash joins whose build side is the big table.
5. Sorts and hashes that spilled to disk.
6. Filters applied after a join that could have run before it.

Cost units are internal and comparable only within one engine. In an interview, narrate the plan: what runs first, where the expensive step is, why the optimiser chose it, and what you would change.

```text
HashAggregate  (rows=200) (actual rows=198)
  ->  Hash Join  (rows=40000) (actual rows=41000)
        Hash Cond: (orders.customer_id = c.customer_id)
        ->  Seq Scan on orders  (rows=40000) (actual rows=41000)
              Filter: (order_date >= '2023-01-01')   Rows Removed by Filter: 159000
        ->  Hash
              ->  Seq Scan on customers  (rows=200) (actual rows=200)
```

Read bottom-up: `customers` is hashed, `orders` is scanned and filtered, the hash join yields 41,000 rows and the aggregate collapses them to 198 groups. Estimates are close to actuals, so this plan is near its natural cost.

The cell below watches SQLite's plan switch from `SCAN` to `SEARCH ... USING INDEX` when an index appears.

In [ ]:
con_lite = sqlite3.connect(":memory:")
con_lite.execute("CREATE TABLE orders (order_id INTEGER PRIMARY KEY, customer_id INTEGER, order_date TEXT, amount REAL)")
con_lite.executemany(
    "INSERT INTO orders VALUES (?, ?, ?, ?)",
    [(i, i % 5000, "2023-01-%02d" % (i % 28 + 1), float(i % 100)) for i in range(1, 50001)],
)


def plan(sql):
    return [row[3] for row in con_lite.execute("EXPLAIN QUERY PLAN " + sql)]


query = "SELECT order_id FROM orders WHERE customer_id = 11"
before = plan(query)
con_lite.execute("CREATE INDEX idx_orders_customer ON orders(customer_id)")
after = plan(query)
print("before index:", before)
print("after index: ", after)
assert any("SCAN" in step for step in before)
assert any("SEARCH" in step and "INDEX" in step for step in after)

### Statistics and plan stability (engine-specific)

The optimiser never looks at the data, only at statistics: row and page counts, distinct values, most-common-value histograms. They go stale after bulk loads, truncate-and-reload, large key-distribution changes or a version upgrade; refresh with `ANALYZE` (Postgres), `UPDATE STATISTICS` (SQL Server) or `DBMS_STATS` (Oracle). Per-column statistics assume independence, so correlated columns (`city`, `state`) are under-estimated; extended statistics (`CREATE STATISTICS ... (dependencies, ndistinct)`) fix that.

A prepared statement may be planned once for all parameters (a generic plan; SQL Server's parameter sniffing) or per value (custom plan). A plan tuned for a small customer is terrible for a whale, so the symptom is a query fast for one parameter and slow for another. Fixes: force a custom plan or `OPTION (RECOMPILE)`, or split the heavy case. Hints (`OPTION (HASH JOIN)`, `pg_hint_plan`, `STRAIGHT_JOIN`) pin one plan forever; use them only after fixing statistics failed, and document why.

### Scan versus seek

A sequential scan reads pages in order, so cost is roughly linear in pages. An index seek can touch a different page per matching row. An index therefore wins at 0.1% selectivity and loses at 40%.

In [ ]:
def scan_cost(pages):
    return pages


def seek_cost(rows_returned, index_height, cache_hit_rate=0.9):
    return index_height + rows_returned * (1 - cache_hit_rate)


PAGES, TABLE_ROWS = 100_000, 10_000_000
for selectivity in (0.001, 0.01, 0.1, 0.4):
    print(f"selectivity={selectivity:>5} scan={scan_cost(PAGES):>7.0f} seek={seek_cost(TABLE_ROWS * selectivity, 4):>9.0f}")
assert seek_cost(TABLE_ROWS * 0.001, 4) < scan_cost(PAGES) < seek_cost(TABLE_ROWS * 0.4, 4)

## Join algorithms and join order

| Algorithm | How it works | Cost | Wins when |
| --- | --- | --- | --- |
| Nested loop | for each outer row, look up the inner side | `outer x log(inner)` with an index, `outer x inner` without | tiny outer side, indexed inner, non-equi predicates, early `LIMIT` |
| Hash join | build a hash table on the smaller input, stream the larger one through it | `build + probe` | large equi-joins without useful order (the normal warehouse shape); spills if the build side exceeds memory |
| Merge join | sort both inputs on the key, walk them in lockstep | `sort + sort + both` (sorts free if inputs are ordered) | both inputs already sorted, range joins, hash would spill |

The optimiser folds a multi-table query into pairwise joins and searches join orders using cardinality estimates (dynamic programming for small queries, greedy or genetic search for large ones). The written order is at most a hint. It prefers selective joins first so intermediate results stay small. That is why filtering a dimension down to ten rows makes the query fast.

**Fan-out**: joining on a key that is not unique on one side multiplies rows (`orders` x `order_items` x `shipments`). The estimator uses `rows(a) * rows(b) / max(ndv_a, ndv_b)`. A query that "returns too many rows" is usually fan-out, a modelling problem; use `EXISTS` when you only need existence, and aggregate before joining.

**Distributed joins** add a data-movement choice: *shuffle* both sides by key, *broadcast* the small side to every worker, or use co-located tables already partitioned on the key. Broadcasting something that is not small (5 GB to 200 workers is 1 TB of network) is a classic failure. **Skew**: one hot key (null customer, test account) sends most rows to one worker, which becomes the straggler. Fix by filtering or isolating the hot key, salting it across N sub-keys, broadcasting, or pre-aggregating.

**Practical rules**: filter dimensions early; join on equality with matching types and collations (an integer-to-string join casts one side and kills the index); refresh statistics after big loads; check the hash build side is the small one; read the plan after every rewrite.

In [ ]:
def nested_loop_cost(outer_rows, inner_rows, inner_indexed):
    return outer_rows * (1 if inner_indexed else inner_rows)


def hash_join_cost(build_rows, probe_rows):
    return build_rows + probe_rows


def join_rows(left_rows, left_ndv, right_rows, right_ndv):
    return left_rows * right_rows / max(left_ndv, right_ndv)


facts, dims = 1_000_000, 50_000
print("nested loop, indexed  :", nested_loop_cost(facts, dims, True))
print("nested loop, unindexed:", nested_loop_cost(facts, dims, False))
print("hash join             :", hash_join_cost(dims, facts))
assert nested_loop_cost(facts, dims, True) < hash_join_cost(dims, facts) < nested_loop_cost(facts, dims, False)

# cardinality: 10M orders over 200k customers; filtering customers to 10% shrinks the join 10x
full = join_rows(10_000_000, 200_000, 200_000, 200_000)
us_only = join_rows(10_000_000, 200_000, 20_000, 20_000)
print("estimated join rows:", round(full), "->", round(us_only))
assert round(full) == 10_000_000 and round(us_only) == 1_000_000

## Indexes and predicates

A B-tree index keeps keys sorted with a short root-to-leaf path: equality is one descent, a range is a descent plus a walk along linked leaves, and an `ORDER BY` on the key comes out pre-sorted. Hash indexes serve equality only; bitmap indexes suit low-cardinality columns. Every index is maintained on every insert, update and delete, so index what you filter, join and sort on, and drop unused ones.

### Sargability

A predicate is **sargable** (Search ARGument ABLE) when the indexed column stands alone on one side of the comparison. Non-sargable patterns: functions on the column (`YEAR(d)`, `UPPER(email)`, `CAST`), arithmetic on the column (`salary * 12 > x`), implicit casts (string column compared to a number), leading wildcards (`LIKE '%smith'`), and `OR` across different columns. Rewrite as a range on the bare column, or build an expression index. The same rule decides partition pruning: `CAST(event_time AS DATE) = '2023-06-01'` scans every partition, while `event_time >= '2023-06-01' AND event_time < '2023-06-02'` prunes to one.

In [ ]:
emp = sqlite3.connect(":memory:")
emp.executescript("""
CREATE TABLE employees (emp_id INTEGER PRIMARY KEY, name TEXT, hire_date TEXT);
INSERT INTO employees VALUES (1, 'Ada', '2022-03-01'), (2, 'Brett', '2023-01-15'), (3, 'Chen', '2023-07-01');
CREATE INDEX idx_employees_hire ON employees(hire_date);
""")


def first_step(sql):
    return emp.execute("EXPLAIN QUERY PLAN " + sql).fetchall()[0][3]


non_sargable = "SELECT name FROM employees WHERE substr(hire_date, 1, 4) = '2023'"
sargable = "SELECT name FROM employees WHERE hire_date >= '2023-01-01' AND hire_date < '2024-01-01'"
print("function on the column:", first_step(non_sargable))
print("range on the column:   ", first_step(sargable))
assert emp.execute(non_sargable).fetchall() == emp.execute(sargable).fetchall()
assert first_step(non_sargable).startswith("SCAN") and "USING INDEX" in first_step(sargable)

### Composite indexes and the leftmost prefix

An index on `(a, b, c)` is sorted by `a`, then `b`, then `c`, like a phone book by last then first name. It can seek on `a`, `(a, b)` or `(a, b, c)`, with a range allowed on the last column used, but not on `b` or `(b, c)` alone. Order columns as **equality first, then the range column, then columns needed only for sorting or covering**. After a range column, later columns can filter inside the index but not seek. Overlapping indexes such as `(a)` and `(a, b)` duplicate work.

In [ ]:
def can_seek(index_columns, predicate_columns):
    """True when the predicate columns form a leftmost prefix of the index."""
    position = 0
    for column in predicate_columns:
        if position < len(index_columns) and index_columns[position] == column:
            position += 1
        else:
            return False
    return position > 0


index = ["customer_id", "order_date", "status"]
for predicate in (["customer_id"], ["customer_id", "order_date"], ["order_date"], ["order_date", "status"]):
    print(predicate, "->", can_seek(index, predicate))
assert can_seek(index, ["customer_id", "order_date", "status"])
assert not can_seek(index, ["order_date"]) and not can_seek(index, ["status"])

### Covering, partial and expression indexes (engine-specific)

- **Covering index**: contains every column the query needs, so the engine answers from the index alone (an index-only scan; in Postgres the visibility map must also mark pages all-visible). `CREATE INDEX ... (customer_id, order_date) INCLUDE (amount)` in SQL Server and Postgres. `SELECT *` almost never qualifies. Cover the two or three hottest queries only, because wide indexes slow every write.
- **Partial (filtered) index**: indexes only rows matching a predicate (`WHERE status = 'open'`), so it is small and cheap to maintain.
- **Expression index**: indexes `lower(email)` or `date_trunc('month', d)` so a function predicate becomes sargable.
- **Unique index**: also tells the planner an equality returns at most one row.
- **Clustered vs secondary**: a clustered index *is* the table in key order (one per table; InnoDB and SQL Server); secondary indexes point back to it and cost an extra lookup per row unless they cover the query. Postgres tables are heaps; `CLUSTER` reorders once. **Bitmap scans** collect matching pages and read them in physical order, turning random reads into sequential ones for medium-selectivity predicates.

### Selectivity

The expected rows for an equality predicate are `rows / distinct_values`. A near-unique column returns about one row, which makes a great index. A four-value `status` column returns a quarter of the table, so the optimiser correctly prefers a scan. An index is also skipped when the table is tiny, the query returns most rows and columns, the types differ, or statistics are stale. Bloated indexes (from heavy updates or deletes) need `REINDEX`, `OPTIMIZE TABLE` or `ALTER INDEX ... REBUILD`.

In [ ]:
TOTAL_ROWS = 1_000_000
distinct_values = {"customer_id": 900_000, "order_date": 1_095, "status": 4}
for column, ndv in distinct_values.items():
    print(f"{column:12s} ndv={ndv:>7,}  rows per equality ~ {TOTAL_ROWS / ndv:>9,.1f}")
best = min(distinct_values, key=lambda c: TOTAL_ROWS / distinct_values[c])
worst = max(distinct_values, key=lambda c: TOTAL_ROWS / distinct_values[c])
assert (best, worst) == ("customer_id", "status")

## Aggregation and window performance

- **Hash vs sort aggregate**: hash is one pass with memory per distinct group; sort aggregate reuses existing order and emits each group when the key changes. Grouping by a near-unique or very wide key makes the hash table itself the problem. Pre-aggregate or coarsen the grain.
- **DISTINCT vs GROUP BY**: planned identically; `DISTINCT` over a wide row is costlier than grouping the columns you need. `COUNT(DISTINCT)` needs its own dedup step and is not supported as a framed window in most engines.
- **Window functions** sort (or redistribute) the entire input by partition and order key. Memory scales with the largest partition, so one skewed partition can spill. Several windows sharing the same `PARTITION BY`/`ORDER BY` share one sort.
- **Reduce rows first**: `WHERE` before `GROUP BY`, aggregate each large side before joining, roll up daily before monthly.
- **Approximate aggregates** (`approx_count_distinct`, HyperLogLog) trade under 1-2% error for far less memory: right for dashboards, wrong for invoices.
- **Spills** show as "external"/"disk" sorts or temp-file counters. Fix by projecting fewer columns before the sort, filtering, pre-aggregating, or raising the memory grant.

In [ ]:
def sort_memory_gib(rows, key_bytes, payload_bytes):
    return rows * (key_bytes + payload_bytes) / 1024 ** 3


for rows in (1_000_000, 100_000_000):
    print(f"rows={rows:>11,}  narrow={sort_memory_gib(rows, 16, 32):6.2f} GiB  wide={sort_memory_gib(rows, 16, 512):6.2f} GiB")
assert round(sort_memory_gib(1_000_000, 16, 32), 2) == 0.04
assert sort_memory_gib(100_000_000, 16, 512) > 10 * sort_memory_gib(100_000_000, 16, 32)

## SQL tuning checklist

1. **Measure**: capture the plan, wall-clock time and input row counts before changing anything.
2. **Read the plan**: find the most expensive operator and the first place estimates diverge from actuals.
3. **Reduce rows early**: push filters to the tables, select only needed columns, ensure partition pruning.
4. **Fix joins**: right algorithm for the sizes, small build side, no accidental fan-out, `EXISTS` for existence.
5. **Make predicates sargable**: no functions or casts on indexed columns; composite indexes equality-then-range; cover hot queries.
6. **Tame memory**: smaller sort and window inputs, pre-aggregation, `UNION ALL` instead of `UNION`, watch for spills.
7. **Refresh statistics** after large loads; add extended statistics for correlated columns.
8. **Re-measure** after each change, keep what helped, roll back what did not.

Example: a nightly report went from 20 seconds to 12 minutes with no code change. Its plan showed a nested loop driven by an estimate of 500 rows (statistics three months old) that actually produced 90 million, each probing an unindexed dimension. Refreshing statistics, indexing the join key, filtering the date range before the join and pre-aggregating brought it back under 30 seconds, and `ANALYZE` was scheduled after every load.

## Real-world SQL use cases

### Signup funnel

Count distinct users per step, not events: `max(CASE ...)` collapses each user's history into one row, so a repeated event is counted once. Counting events instead of users is the classic funnel bug.

In [ ]:
con.execute("""
CREATE TEMP TABLE funnel_events AS SELECT * FROM (VALUES
    (1, 'landing'), (1, 'signup'), (1, 'purchase'), (1, 'purchase'),
    (2, 'landing'), (2, 'signup'),
    (3, 'landing'), (3, 'landing')
) t(user_id, event_name)
""")
funnel = q("""
    WITH per_user AS (
        SELECT user_id,
               max(CASE WHEN event_name = 'landing'  THEN 1 ELSE 0 END) AS landed,
               max(CASE WHEN event_name = 'signup'   THEN 1 ELSE 0 END) AS signed_up,
               max(CASE WHEN event_name = 'purchase' THEN 1 ELSE 0 END) AS purchased
        FROM funnel_events GROUP BY user_id
    )
    SELECT sum(landed), sum(signed_up), sum(purchased),
           round(100.0 * sum(purchased) / sum(landed), 1) AS conversion_pct
    FROM per_user
""")
print(funnel)
assert funnel == [(3, 2, 1, 33.3)]

### Monthly cohort retention

Assign each user a signup month, compute `month_number` as the month difference between activity and signup, then count distinct users per cohort and month. `count(DISTINCT user_id) FILTER (WHERE month_number = 1)` (Postgres, Snowflake, DuckDB) avoids a self-join per period.

In [ ]:
con.execute("""
CREATE TEMP TABLE activity AS SELECT * FROM (VALUES
    (1, DATE '2026-01-05', DATE '2026-01-05'), (1, DATE '2026-01-05', DATE '2026-02-10'),
    (1, DATE '2026-01-05', DATE '2026-03-01'), (2, DATE '2026-01-20', DATE '2026-02-02'),
    (3, DATE '2026-02-01', DATE '2026-02-01'), (3, DATE '2026-02-01', DATE '2026-03-15')
) t(user_id, signup_date, activity_date)
""")
cohorts = q("""
    WITH a AS (
        SELECT date_trunc('month', signup_date) AS cohort, user_id,
               datediff('month', date_trunc('month', signup_date), date_trunc('month', activity_date)) AS month_number
        FROM activity
    )
    SELECT strftime(cohort, '%Y-%m'),
           count(DISTINCT user_id)                                     AS cohort_size,
           count(DISTINCT user_id) FILTER (WHERE month_number = 1)    AS month_1,
           count(DISTINCT user_id) FILTER (WHERE month_number = 2)    AS month_2
    FROM a GROUP BY cohort ORDER BY cohort
""")
print(cohorts)
assert cohorts == [("2026-01", 2, 2, 1), ("2026-02", 1, 1, 0)]

### Daily revenue mart

Finance needs per-day, per-country revenue that reconciles with the ledger. The fact table is partitioned by `order_date`; the job rebuilds only the last two days to absorb late refunds, so it scans gigabytes rather than terabytes. `count(DISTINCT order_id)` survives the fan-out from joining order lines, and an explicit `CASE` keeps refunds negative instead of dropping them.

```sql
INSERT OVERWRITE mart_daily_revenue PARTITION (order_date)      -- or DELETE + INSERT for the window
SELECT o.order_date, o.country_code,
       count(DISTINCT o.order_id)                                   AS orders,
       sum(CASE WHEN o.status = 'refunded' THEN -1 ELSE 1 END
           * oi.quantity * oi.unit_price)                           AS net_revenue
FROM fact_orders o
JOIN fact_order_lines oi ON oi.order_id = o.order_id AND oi.order_date = o.order_date
WHERE o.order_date BETWEEN :run_date - INTERVAL '1' DAY AND :run_date
GROUP BY o.order_date, o.country_code;
```

### Running balance with top-N in one scan

For a ledger report, `sum(amount) OVER (PARTITION BY customer_id ORDER BY posting_date, ledger_id ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)` gives the running balance and `row_number() OVER (PARTITION BY customer_id ORDER BY amount DESC, ledger_id)` ranks transactions. Both share one scan of the partition-pruned range. The explicit `ROWS` frame and the `ledger_id` tiebreaker keep the balance correct when several postings share a date.

## Practice: SQL interview questions

The hands-on practice for this topic lives in its own notebook: [`03_sql_interview_questions.ipynb`](sql-00-start-here.ipynb) — [Open in Colab](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/sql/sql-00-start-here.ipynb).